# RAG sobre PDF (paso a paso, sin LangChain)

Este notebook usa los módulos de `src/`:

1. Extracción de texto del PDF (`pypdf`)
2. Chunking recursivo manual
3. Embeddings locales (`sentence-transformers` / Hugging Face)
4. Vector DB **Qdrant** (inspección en [dashboard](http://localhost:6333/dashboard))
5. Recuperación + prompt con contexto
6. Generación con **Groq**
7. Comparativa **con RAG vs sin RAG**

**Prerrequisitos:** `docker compose up -d`, `GROQ_API_KEY` en `.env`, PDF de ejemplo en `data/`.

## 0. Configurar el path del proyecto

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent  # si abriste el notebook desde notebooks/
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.chdir(ROOT)
print("ROOT =", ROOT)

## 1. Comprobar Qdrant

Si falla, levantá el contenedor: `docker compose up -d` en la raíz del repo.

In [ ]:
import urllib.request
from src import config

health_url = config.QDRANT_URL.rstrip("/") + "/readyz"
with urllib.request.urlopen(health_url, timeout=2) as r:
    print("Qdrant OK:", r.status, health_url)

## 2. Cargar el PDF

In [ ]:
from src.pdf_loader import extract_text_by_page

PDF_PATH = ROOT / "data" / "ejemplo.pdf"  # reemplazá por tu PDF
if not PDF_PATH.exists():
    raise FileNotFoundError(f"Poné un PDF en {PDF_PATH} o cambiá PDF_PATH.")

pages = extract_text_by_page(PDF_PATH)
print("Páginas:", len(pages))
sample = (pages[0].get("text") or "")[:500]
print("Primeros 500 caracteres de la página 1:\n", sample)

## 3. Chunking recursivo

In [ ]:
from src.chunker import chunk_pages
from src import config

chunks = chunk_pages(
    pages,
    chunk_size=config.CHUNK_SIZE,
    chunk_overlap=config.CHUNK_OVERLAP,
    source=str(PDF_PATH),
)
print("Total chunks:", len(chunks))
for c in chunks[:2]:
    print("--- chunk", c["chunk_index"], "page", c["page"], "---")
    print(c["text"][:400], "..." if len(c["text"]) > 400 else "")

## 4. Embeddings (forma del vector)

In [ ]:
from src.embedder import Embedder

embedder = Embedder()
one = embedder.encode([chunks[0]["text"]])
print("shape:", one.shape, "dtype:", one.dtype)
print("primeros 10 valores:", one[0, :10])

## 5. Indexar en Qdrant y mirar puntos

Abrí el [dashboard](http://localhost:6333/dashboard) y revisá la colección `pdf_chunks`.

In [ ]:
from src.vector_store import QdrantStore

store = QdrantStore()
store.delete_collection()
store.ensure_collection()

texts = [c["text"] for c in chunks]
vectors = embedder.encode(texts, show_progress_bar=True)
print("vectors:", vectors.shape)

store.upsert_chunks(chunks, vectors, start_id=0)
print("count:", store.count())

for rec in store.peek(limit=3):
    print("id:", rec.id, "payload keys:", list((rec.payload or {}).keys()))
    pv = rec.vector
    if isinstance(pv, dict):
        print("vector (named):", list(pv.keys()))
    else:
        print("vector len:", len(pv) if pv is not None else None)

## 6. Recuperación (query → top-k)

In [ ]:
pregunta = "Resumí la idea principal del documento."  # editá a gusto

q_vec = embedder.encode([pregunta])
print("query shape:", q_vec.shape)


## 7. Generar respuesta con contexto (RAG)

In [ ]:
from src import config
from src.rag import SYSTEM_PROMPT
from src.llm import GroqLLM

hits = store.search(q_vec, top_k=config.TOP_K)
for i, h in enumerate(hits, start=1):
    print(i, "score=", round(h.score, 4), "page=", h.payload.get("page"))

context_blocks = []
for i, h in enumerate(hits, start=1):
    context_blocks.append(
        f"[Fragmento {i} | página {h.payload.get('page')} | score {h.score:.4f}]\n"
        f"{h.payload.get('text', '')}\n---"
    )
context = "\n".join(context_blocks)

user_message = (
    "Contexto recuperado del documento:\n\n" + context + "\n\nPregunta: " + pregunta
)

llm = GroqLLM()
answer = llm.chat(SYSTEM_PROMPT, user_message)
print("=== Prompt user (preview) ===")
print(user_message[:1200], "..." if len(user_message) > 1200 else "")
print("\n=== Respuesta ===\n")
print(answer)

## 8. Comparativa: sin RAG vs con RAG

In [ ]:
from src.rag import answer_without_rag

print("--- Sin RAG (solo LLM) ---")
print(answer_without_rag(pregunta))

print("\n--- Con RAG (arriba) ---")
print(answer)